World Cup 2026 Predictor

A ML project that predicts the outcomes of FIFA World Cup 2026 matches and simulates the tournament bracket to crown a predicted winner from player performance data.

The dataset contains per-player, per-match performance statistics (~54,600 rows, 75 features). To predict match outcomes, individual player rows are aggregated up to the team-vs-team match level, then a classifier is trained to predict the result.

In [5]:
!pip install kagglehub
!pip install seaborn
!pip install shap

import os
import pandas as pd
import numpy as np
import kagglehub
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_selection import RFECV
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer

<div style="background-color: #333333; color: #FFFFFF; padding: 15px; border-radius: 5px;">
<span style="font-size: 24px;">1. Beginning</span>

</div>

In [4]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

path = "fifa_world_cup_2026_player_performance.csv"

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "rauffauzanrambe/fifa-world-cup-2026-player-performance-dataset",
    path,
)
         
df.head()               

,player_id,player_name,age,nationality,team,jersey_number,position,height_cm,weight_kg,preferred_foot,...,possession_impact,pressure_resistance,creativity_score,consistency_score,clutch_performance_score,total_goals_tournament,total_assists_tournament,total_minutes_tournament,player_of_match_awards,tournament_rating
0,P00055,Rodri Fati,26,Spanish,Spain,3,Goalkeeper,195,75,Left,...,1.1,44.2,55.9,42.0,51.8,0,0,242,0,5.8
1,P00070,Ansu Le Normand,19,Spanish,Spain,18,Midfielder,178,75,Right,...,3.5,38.2,43.7,31.1,52.7,0,3,342,0,5.5
2,P00066,Gavi Ramos,18,Spanish,Spain,14,Midfielder,177,72,Left,...,15.3,99.0,99.0,83.4,54.8,1,1,245,0,8.4
3,P00073,Pedro Cubarsi,20,Spanish,Spain,21,Forward,182,74,Right,...,1.2,19.8,42.3,40.9,78.5,5,3,422,0,6.7
4,P00059,Alvaro Oyarzabal,23,Spanish,Spain,7,Defender,191,81,Left,...,6.2,44.1,33.5,60.0,56.6,0,0,440,0,5.7


<div style="background-color: #333333; color: #FFFFFF; padding: 15px; border-radius: 5px;">
<span style="font-size: 24px;">2. Data Splitting</span>

I split the data into 70% training data and 30% test data with random_state = 123

</div>

In [6]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=123)

<div style="background-color: #333333; color: #FFFFFF; padding: 15px; border-radius: 5px;">
<span style="font-size: 24px;">3. EDA</span>
    
A) Perform exploratory data analysis (EDA): Conduct an initial exploration of the training set to better understand its characteristics.

B) Summarize and visualize the data: Include at least two summary statistics and two visualizations that you find informative. For each, write one sentence explaining what insight it provides.

C) Record your observations: Summarize your initial observations about the dataset based on your EDA.

D) Select evaluation metrics: Choose one or more appropriate metrics for assessing model performance and briefly justify your choice
</div>

In [7]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 38220 entries, 7369 to 52734
Data columns (total 75 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   player_id                 38220 non-null  object 
 1   player_name               38220 non-null  object 
 2   age                       38220 non-null  int64  
 3   nationality               38220 non-null  object 
 4   team                      38220 non-null  object 
 5   jersey_number             38220 non-null  int64  
 6   position                  38220 non-null  object 
 7   height_cm                 38220 non-null  int64  
 8   weight_kg                 38220 non-null  int64  
 9   preferred_foot            38220 non-null  object 
 10  club_name                 38220 non-null  object 
 11  market_value_eur          38220 non-null  int64  
 12  match_id                  38220 non-null  object 
 13  match_date                38220 non-null  object 
 14  stadium 

A) The training dataset has 38220 players and 75 features.  

In [8]:
train_df.describe()

,age,jersey_number,height_cm,weight_kg,market_value_eur,goals_team,goals_opponent,minutes_played,goals,assists,...,possession_impact,pressure_resistance,creativity_score,consistency_score,clutch_performance_score,total_goals_tournament,total_assists_tournament,total_minutes_tournament,player_of_match_awards,tournament_rating
count,38220.000000,38220.000000,38220.000000,38220.000000,3.822000e+04,38220.000000,38220.000000,38220.000000,38220.000000,38220.000000,...,38220.000000,38220.000000,38220.000000,38220.000000,38220.000000,38220.000000,38220.000000,38220.000000,38220.000000,38220.000000
mean,26.305861,13.458687,181.661355,75.760440,2.001953e+07,1.329906,1.331293,36.200942,0.054605,0.051910,...,2.850018,60.320243,46.076839,63.719584,55.590740,0.635479,0.606096,271.801282,0.030167,3.635502
std,4.054208,7.505770,6.264343,3.943167,2.710294e+07,1.150075,1.145378,36.384504,0.250959,0.238117,...,4.244032,20.230034,22.381039,19.824896,13.659334,1.081681,0.929538,116.616298,0.202435,3.162640
min,17.000000,1.000000,163.000000,65.000000,5.288220e+05,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,15.000000,5.000000,25.000000,15.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,23.000000,7.000000,177.000000,73.000000,4.416108e+06,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,43.700000,29.800000,46.600000,46.300000,0.000000,0.000000,184.000000,0.000000,0.000000
50%,26.000000,13.000000,182.000000,76.000000,1.024334e+07,1.000000,1.000000,25.000000,0.000000,0.000000,...,1.100000,57.000000,40.500000,61.000000,55.100000,0.000000,0.000000,267.000000,0.000000,5.400000
75%,29.000000,20.000000,186.000000,78.000000,2.338162e+07,2.000000,2.000000,75.000000,0.000000,0.000000,...,4.000000,75.000000,58.800000,78.800000,63.900000,1.000000,1.000000,358.000000,0.000000,6.400000
max,39.000000,26.000000,200.000000,87.000000,2.000000e+08,7.000000,7.000000,90.000000,4.000000,3.000000,...,34.700000,99.000000,99.000000,99.000000,99.000000,10.000000,8.000000,601.000000,4.000000,9.400000


<div style="background-color: #333333; color: #FFFFFF; padding: 15px; border-radius: 5px;">
<span style="font-size: 24px;">4. Feature Engineering</span>

Create new features that are relevant to the problem and use this updated feature set in the following exercises.

</div>

In [9]:
# train_df['AVG_BILL'] = train_df[['BILL_AMT1','BILL_AMT2','BILL_AMT3',
#                                  'BILL_AMT4','BILL_AMT5','BILL_AMT6']].mean(axis=1)

# train_df['CREDIT_USE'] = train_df['AVG_BILL'] / train_df['LIMIT_BAL']

<div style="background-color: #333333; color: #FFFFFF; padding: 15px; border-radius: 5px;">
<span style="font-size: 24px;">5. Preprocessing and transformations</span>

A) Identify feature types: Determine the different types of features in your dataset (e.g., numerical, categorical, ordinal, text) and specify the transformations you would apply to each type.

B) Define a column transformer (if needed): Implement a ColumnTransformer to apply the appropriate preprocessing steps to each feature type.
</div>

<div style="background-color: #333333; color: #FFFFFF; padding: 15px; border-radius: 5px;">
<span style="font-size: 24px;">6. Baseline Model</span>

Establish a baseline: Use one of scikit-learn’s baseline models (e.g., DummyClassifier or DummyRegressor, depending on your task) and report the results. This will serve as a reference point for evaluating the performance of your more advanced models.

</div>

<div style="background-color: #333333; color: #FFFFFF; padding: 15px; border-radius: 5px;">
<span style="font-size: 24px;">7. Linear Models</span>

Train a linear model: Use a linear model as your first real attempt at solving the problem.

Tune hyperparameters: Perform hyperparameter tuning to explore different values of the model's complexity parameter.

Evaluate with cross-validation: Report the cross-validation scores along with their standard deviation.

Summarize findings: Summarize your results, highlighting key observations from your experiments.

</div>

<div style="background-color: #333333; color: #FFFFFF; padding: 15px; border-radius: 5px;">
<span style="font-size: 24px;">8. Different Models</span>

A) Experiment with additional models: Train at least three models other than a linear model. Ensure that at least one of these models is a tree-based ensemble model (e.g., Random Forest, Gradient Boosting, or XGBoost).

B) Compare and interpret results: Summarize your findings in terms of overfitting/underfitting behavior and fit/score times for each model. Reflect on your results. Were you able to outperform the linear model?
</div>

<div style="background-color: #333333; color: #FFFFFF; padding: 15px; border-radius: 5px;">
<span style="font-size: 24px;">9. Feature Selection</span>

A) Perform feature selection: Attempt to select relevant features using methods such as RFECV or forward selection.

B) Evaluate the impact Compare the model performance before and after feature selection. Do the results improve with feature selection?

C) Summarize findings Summarize your observations and decide whether to keep feature selection in your pipeline. If it improves results, retain it for the next exercises; otherwise, you may choose to omit it.
</div>

<div style="background-color: #333333; color: #FFFFFF; padding: 15px; border-radius: 5px;">
<span style="font-size: 24px;">10. Hyperparameter Optimization</span>

A) Optimize hyperparameters: Attempt to optimize hyperparameters for the models you have tried so far. In at least one case, tune multiple hyperparameters for a single model.

B) Use suitable optimization methods: You may use any of the following approaches for hyperparameter optimization:
- GridSearchCV
- RandomizedSearchCV
- Bayesian optimization with scikit-optimize

C) Summarize your results: Report and compare the optimized results across models. Discuss whether hyperparameter optimization led to performance improvements.
</div>

<div style="background-color: #333333; color: #FFFFFF; padding: 15px; border-radius: 5px;">
<span style="font-size: 24px;">11. Interpretation and feature importances</span>

A) Interpret model feature importance: Use one of the interpretation methods discussed in class (e.g., shap), or another suitable method of your choice, to examine the most important features of one of your non-linear models.

B) Summarize insights: Summarize your observations about which features contribute most to the model's predictions and how they influence the outcomes.
</div>

<div style="background-color: #333333; color: #FFFFFF; padding: 15px; border-radius: 5px;">
<span style="font-size: 24px;">12. Results on the test set</span>

A) Evaluate on the test set: Apply your best-performing model to the test data and report the test scores.

B) Compare and reflect: Compare the test scores with the validation scores from previous experiments. Discuss the consistency between them. How much do you trust your results? Reflect on whether you might have encountered optimization bias.

C) Explain individual predictions: Select one or two examples from your test predictions and use an interpretation method (e.g., SHAP force plots) to explain these individual predictions.
</div>

<div style="background-color: #333333; color: #FFFFFF; padding: 15px; border-radius: 5px;">
<span style="font-size: 24px;">13. Summary</span>

A) Summarize key results: Create a clear and concise table highlighting your most important results (e.g., models compared, validation/test scores, key observations).

B) Write concluding remarks: Summarize your main takeaways from the project, including what worked well and what did not.

C) Propose future improvements: Discuss ideas or approaches you did not try but that could potentially improve performance or interpretability.

Report final results: Report your final test score and the metric you used.
</div>